# Episode 2: Pretraining a GPT from scratch in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep002-pretraining.ipynb)

From random tokens to short stories in a first training run on a laptop, with a tokenizer, the next-token objective, a training loop and evaluation written from scratch.

[The video](https://youtu.be/KA0aVTQ5Wak) · [The episode's page](https://mvgpt.nenuadrian.com/ep002-pretraining/) · [The docs](https://nenuadrian.github.io/mvGPT/pretraining/) · [The code at `ep002/end`](https://github.com/nenuadrian/mvGPT/tree/ep002/end)

This notebook runs the episode's commands on its code, as the episode left it. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep002/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## Pretraining
Ten minutes of training; the episode's run on a laptop took 1,250 steps. A checkpoint every
250 steps goes in `checkpoints/pretrain/`.

In [ ]:
!python pretrain.py 10

A story from the same prompt at each evaluation, from the run's log:

In [ ]:
import json

for e in json.load(open("checkpoints/pretrain/log.json"))["evals"]:
    print(f"step {e['step']}: {e['sample']}\n")

## Writing stories

In [ ]:
!python sample.py checkpoints/pretrain/model.pt
!python sample.py checkpoints/pretrain/step-250.pt -n 1
!python sample.py checkpoints/pretrain/model.pt "Tom and his dog" -n 1 --temperature 1.6

## Measuring the checkpoints
Loss, perplexity and accuracy on held-out stories, and the likeliest next tokens, for each
checkpoint, from the first to the last.

In [ ]:
import glob

steps = sorted(glob.glob("checkpoints/pretrain/step-*.pt"), key=lambda p: int(p.split("-")[-1][:-3]))
!python evaluate.py {" ".join(steps)} checkpoints/pretrain/model.pt

Next: [episode 3, SFT: fine-tuning a GPT to follow instructions](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep003-sft.ipynb)